# Seguridad de la API REST

El marco **SAIF** (Secure AI Framework) aplicado al servicio real `tp_mlops2.api.main`,
que corre en `http://localhost:8000` dentro de Docker Compose.

A diferencia de una demostracion aislada, los controles de este notebook no viven aca:
viven en `src/tp_mlops2/api/security.py` y estan activos en el contenedor `api` todo el
tiempo. Este notebook llama al servicio real por HTTP para probarlos, igual que lo haria
cualquier cliente externo.

In [1]:
import requests

BASE_URL = "http://localhost:8000"
API_KEY = "demo-key-tp-mlops2"  # la misma MODELO_API_KEY del .env del proyecto

muestra = {
    "timestamp": "2018-06-15T14:00:00",
    "temp_madrid": 28.0, "temp_barcelona": 25.0, "temp_valencia": 27.0,
    "temp_seville": 32.0, "temp_bilbao": 20.0,
    "load_24h_ago": 30000.0, "load_168h_ago": 29500.0,
}

print("health:", requests.get(f"{BASE_URL}/health", timeout=10).json())

health: {'status': 'ok'}


## 1. Sin credenciales: rechazado

`/v1/predict` exige la API key `X-API-Key`. Sin ella, el servicio real responde 401,
no deja pasar nada.

In [2]:
r_sin_key = requests.post(f"{BASE_URL}/v1/predict", json=muestra, timeout=10)
print("sin API key:", r_sin_key.status_code, "(esperado 401)")

sin API key: 401 (esperado 401)


## 2. Con API key: predice sobre el modelo real

In [3]:
headers = {"X-API-Key": API_KEY}
r_ok = requests.post(f"{BASE_URL}/v1/predict", json=muestra, headers=headers, timeout=10)
print("con API key:", r_ok.status_code, r_ok.json())

con API key: 200 {'predicted_load_mw': 30694.077109585807, 'timestamp': '2018-06-15T14:00:00', 'model_version': 'random_forest_demanda v8'}


## 3. Payload invalido: Pydantic lo rechaza antes de llegar al modelo

In [4]:
payload_invalido = {"timestamp": "no-es-una-fecha"}
r_422 = requests.post(f"{BASE_URL}/v1/predict", json=payload_invalido, headers=headers, timeout=10)
print("payload invalido:", r_422.status_code, "(esperado 422)")

payload invalido: 422 (esperado 422)


## 4. Rate limiting: la API corta despues del limite por minuto

In [5]:
from tp_mlops2.api.security import LIMITE_POR_MINUTO

codigos = [
    requests.post(f"{BASE_URL}/v1/predict", json=muestra, headers=headers, timeout=10).status_code
    for _ in range(LIMITE_POR_MINUTO + 3)
]
print(f"limite configurado: {LIMITE_POR_MINUTO} requests/minuto")
print("ultimos codigos:", codigos[-5:], "(los que exceden el limite dan 429)")

limite configurado: 60 requests/minuto
ultimos codigos: [429, 429, 429, 429, 429] (los que exceden el limite dan 429)


## 5. Metricas sensibles: solo con JWT y rol autorizado

`/v1/model/metrics` no se protege con API key sino con un JWT por rol, porque es un
endpoint distinto: cualquier cliente integrado puede predecir, pero las metricas internas
del modelo (MAE, MAPE) solo deberian verlas roles de `admin` o `analista`.

In [6]:
from tp_mlops2.auth import crear_token

token_analista = crear_token("alan", "analista")
token_invitado = crear_token("visitante", "invitado")

r_sin_token = requests.get(f"{BASE_URL}/v1/model/metrics", timeout=10)
r_rol_insuficiente = requests.get(
    f"{BASE_URL}/v1/model/metrics",
    headers={"Authorization": f"Bearer {token_invitado}"}, timeout=10,
)
r_ok = requests.get(
    f"{BASE_URL}/v1/model/metrics",
    headers={"Authorization": f"Bearer {token_analista}"}, timeout=10,
)

print("sin token    :", r_sin_token.status_code, "(esperado 401)")
print("rol invitado :", r_rol_insuficiente.status_code, "(esperado 403)")
print("rol analista :", r_ok.status_code, r_ok.json())

sin token    : 401 (esperado 401)
rol invitado : 403 (esperado 403)
rol analista : 200 {'mae_mw': 1736.815230933058, 'mape': 0.05999729717375087}


## Reflexion

Los cinco controles -API key, validacion de esquema, rate limiting, JWT por rol y la
auditoria que registra cada intento- estan activos en el contenedor `api` todo el tiempo,
no solo durante esta demostracion. Lo que cambio frente a la version anterior del proyecto
no es el control en si, sino donde vive: antes solo existia en este notebook contra un
servicio de prueba sin seguridad; ahora el servicio real que corre en produccion es el que
tiene el control, y este notebook lo prueba desde afuera, como cualquier cliente.